## NCAA Tournament Simulation with Elo Ratings
This script calculates Elo ratings for both Men's and Women's NCAA basketball teams based on their performance in the 2024 season. It then uses these ratings to simulate the outcomes of the 2023 NCAA tournament games for both Men's and Women's brackets. The simulation predicts winners for each game based on Elo ratings, and the results are prepared into a single CSV file suitable for submission.

In [1]:
import numpy as np
import pandas as pd 

In [2]:
# Define functions to update Elo ratings and predict game outcomes
def update_elo(winner_rating, loser_rating):
    """
    Update Elo ratings after a game.

    Parameters:
    - winner_rating (float): The Elo rating of the winning team before the game.
    - loser_rating (float): The Elo rating of the losing team before the game.

    Returns:
    - new_winner_rating (float): Updated Elo rating of the winning team.
    - new_loser_rating (float): Updated Elo rating of the losing team.
    """
    K = 20  # Maximum change per game
    expected_win = 1 / (1 + 10 ** ((loser_rating - winner_rating) / 400))
    new_winner_rating = winner_rating + K * (1 - expected_win)
    new_loser_rating = loser_rating - K * (1 - expected_win)
    return new_winner_rating, new_loser_rating

def predict_winner(team1_id, team2_id, elo_df):
    """
    Predict the game winner based on Elo ratings.

    Parameters:
    - team1_id (int): The ID of the first competing team.
    - team2_id (int): The ID of the second competing team.
    - elo_df (pandas.DataFrame): DataFrame containing teams' Elo ratings.

    Returns:
    - The ID of the predicted winning team based on Elo ratings.
    """
    elo1 = elo_df.loc[elo_df['TeamID'] == team1_id, 'EloRating'].iloc[0]
    elo2 = elo_df.loc[elo_df['TeamID'] == team2_id, 'EloRating'].iloc[0]
    return team1_id if elo1 > elo2 else team2_id

def simulate_tournament(seeds_df, slots_df):
    """
    Simulate tournament games and determine winners for each slot.

    Parameters:
    - seeds_df (pandas.DataFrame): DataFrame containing teams' seeds and IDs, enriched with Elo ratings.
    - slots_df (pandas.DataFrame): DataFrame detailing tournament structure and matchups.

    Returns:
    - slot_winners (dict): A dictionary mapping each tournament slot to the winning team's seed.
    """
    slot_winners = {}   # Initialize a dictionary to store winners of each slot
    for _, row in slots_df.iterrows():
        slot = row['Slot'] # Identify the current slot
        
        # Round 1 uses direct seeds; subsequent rounds use winners from previous rounds
        if row['Slot'].startswith('R1'):
            strong_team_seed = row['StrongSeed']
            weak_team_seed = row['WeakSeed']
        else:
            strong_team_seed = slot_winners.get(row['StrongSeed'])
            weak_team_seed = slot_winners.get(row['WeakSeed'])

        # Get team IDs and predict the winner
        strong_team_id = seeds_df.loc[seeds_df['Seed'] == strong_team_seed, 'TeamID'].iloc[0]
        weak_team_id = seeds_df.loc[seeds_df['Seed'] == weak_team_seed, 'TeamID'].iloc[0]
        winner_id = predict_winner(strong_team_id, weak_team_id, seeds_df)
        winner_seed = seeds_df.loc[seeds_df['TeamID'] == winner_id, 'Seed'].iloc[0]
        slot_winners[slot] = winner_seed

    return slot_winners

In [3]:
def calculate_elo_ratings(games_df, tournament_type):
    """
    Calculate Elo ratings from the games DataFrame.
    :param games_df: DataFrame containing the games for the season.
    :param tournament_type: 'M' for Men's and 'W' for Women's tournaments.
    :return: DataFrame with TeamID and EloRating columns.
    """
    team_ratings = {team: 1500 for team in set(games_df['WTeamID']).union(games_df['LTeamID'])}
    for _, row in games_df.iterrows():
        w_rating, l_rating = team_ratings[row['WTeamID']], team_ratings[row['LTeamID']]
        team_ratings[row['WTeamID']], team_ratings[row['LTeamID']] = update_elo(w_rating, l_rating)
    
    return pd.DataFrame(list(team_ratings.items()), columns=['TeamID', 'EloRating'])

def merge_elo_with_seeds(seeds_df, elo_df):
    """
    Merge the seeds DataFrame with the Elo ratings DataFrame.
    :param seeds_df: DataFrame containing the tournament seeds.
    :param elo_df: DataFrame containing Elo ratings for teams.
    :return: Merged DataFrame including seeds and Elo ratings.
    """
    seeds_df = seeds_df.copy()
    seeds_df['TeamID'] = seeds_df['TeamID'].astype(int)
    return pd.merge(seeds_df, elo_df, on='TeamID', how='left')

In [4]:
def prepare_submission_data(winners, tournament_type, start_row_id=1):
    """
    Prepare the submission data for a single tournament, starting RowId from given start_row_id.
    
    Parameters:
    - winners (dict): Dictionary of slot winners from simulate_tournament function.
    - tournament_type (str): 'M' for Men's tournament or 'W' for Women's tournament.
    - start_row_id (int): The starting RowId for this tournament's predictions.
    
    Returns:
    - A list of dictionaries with RowId, Tournament, Bracket, Slot, and Team for each winner.
    """
    submission_data = []
    row_id = start_row_id

    for slot, winner_seed in winners.items():
        game_data = {
            'RowId': row_id,
            'Tournament': tournament_type,
            'Bracket': 1,  # Assuming a single bracket simulation
            'Slot': slot,
            'Team': winner_seed
        }
        submission_data.append(game_data)
        row_id += 1

    return submission_data, row_id  # Return the updated row_id for subsequent use

In [6]:
# Load datasets
YEAR = 2024
DATA_DIR = f'../../data/{YEAR}/'
df_games = pd.read_csv(f'{DATA_DIR}MRegularSeasonCompactResults.csv')
df_games_w = pd.read_csv(f'{DATA_DIR}WRegularSeasonCompactResults.csv')
df_seeds = pd.read_csv(f'{DATA_DIR}2024_tourney_seeds.csv')
round_slots = pd.read_csv(f'{DATA_DIR}MNCAATourneySlots.csv')
round_slots_w = pd.read_csv(f'{DATA_DIR}WNCAATourneySlots.csv')

In [7]:
# Filter and exclude play-in games
round_slots = round_slots.loc[(round_slots['Season'] == 2023) & (round_slots['Slot'].str.startswith('R'))]
round_slots_w = round_slots_w.loc[(round_slots_w['Season'] == 2023) & (round_slots_w['Slot'].str.startswith('R'))]

In [8]:
# Calculate and merge Elo ratings for Men's
elo_df_m = calculate_elo_ratings(df_games[df_games['Season'] == 2024], 'M')
df_seeds_m = merge_elo_with_seeds(df_seeds[df_seeds['Tournament'] == 'M'], elo_df_m)

In [9]:
df_seeds_m

,Tournament,Seed,TeamID,EloRating
0,M,W01,1163,1709.078366
1,M,W02,1235,1667.421915
2,M,W03,1228,1645.301820
3,M,W04,1120,1658.235629
4,M,W05,1361,1595.552448
...,...,...,...,...
59,M,Z12,1241,1665.157187
60,M,Z13,1436,1644.941511
61,M,Z14,1324,1590.077150
62,M,Z15,1443,1543.195370


In [10]:
# Calculate and merge Elo ratings for Women's
elo_df_w = calculate_elo_ratings(df_games_w[df_games_w['Season'] == 2024], 'W')
df_seeds_w = merge_elo_with_seeds(df_seeds[df_seeds['Tournament'] == 'W'], elo_df_w)

In [11]:
df_seeds_w

,Tournament,Seed,TeamID,EloRating
0,W,W01,3376,1748.987147
1,W,W02,3323,1671.874515
2,W,W03,3333,1634.865307
3,W,W04,3231,1643.106096
4,W,W05,3328,1620.747124
...,...,...,...,...
59,W,Z12,3162,1634.279877
60,W,Z13,3267,1649.148242
61,W,Z14,3238,1619.738361
62,W,Z15,3263,1611.326898


In [14]:
men_winners

{'R1W1': 'W01',
 'R1W2': 'W02',
 'R1W3': 'W03',
 'R1W4': 'W04',
 'R1W5': 'W05',
 'R1W6': 'W11',
 'R1W7': 'W10',
 'R1W8': 'W08',
 'R1X1': 'X01',
 'R1X2': 'X02',
 'R1X3': 'X03',
 'R1X4': 'X13',
 'R1X5': 'X12',
 'R1X6': 'X11',
 'R1X7': 'X10',
 'R1X8': 'X08',
 'R1Y1': 'Y01',
 'R1Y2': 'Y02',
 'R1Y3': 'Y03',
 'R1Y4': 'Y13',
 'R1Y5': 'Y12',
 'R1Y6': 'Y06',
 'R1Y7': 'Y10',
 'R1Y8': 'Y08',
 'R1Z1': 'Z01',
 'R1Z2': 'Z02',
 'R1Z3': 'Z03',
 'R1Z4': 'Z13',
 'R1Z5': 'Z12',
 'R1Z6': 'Z06',
 'R1Z7': 'Z07',
 'R1Z8': 'Z08',
 'R2W1': 'W01',
 'R2W2': 'W02',
 'R2W3': 'W03',
 'R2W4': 'W04',
 'R2X1': 'X01',
 'R2X2': 'X10',
 'R2X3': 'X11',
 'R2X4': 'X12',
 'R2Y1': 'Y01',
 'R2Y2': 'Y02',
 'R2Y3': 'Y06',
 'R2Y4': 'Y12',
 'R2Z1': 'Z01',
 'R2Z2': 'Z02',
 'R2Z3': 'Z03',
 'R2Z4': 'Z12',
 'R3W1': 'W01',
 'R3W2': 'W02',
 'R3X1': 'X01',
 'R3X2': 'X10',
 'R3Y1': 'Y01',
 'R3Y2': 'Y06',
 'R3Z1': 'Z01',
 'R3Z2': 'Z02',
 'R4W1': 'W01',
 'R4X1': 'X01',
 'R4Y1': 'Y01',
 'R4Z1': 'Z01',
 'R5WX': 'W01',
 'R5YZ': 'Z01',
 'R6CH':

In [12]:
# Simulate Men's and Women's tournaments
men_winners = simulate_tournament(df_seeds_m, round_slots)
women_winners = simulate_tournament(df_seeds_w, round_slots_w)

In [13]:
# Start with RowId 1 for Men's tournament
men_submission_data, next_row_id = prepare_submission_data(men_winners, 'M', 1)

# Continue with the next RowId for Women's tournament
women_submission_data, _ = prepare_submission_data(women_winners, 'W', next_row_id)

# Combine Men's and Women's submission data
combined_submission_data = men_submission_data + women_submission_data

In [ ]:
# Convert to DataFrame
df_submission = pd.DataFrame(combined_submission_data)
df_submission.to_csv('submission.csv', index=False)

In [ ]:
df_submission